[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/22_conv2d_solution.ipynb)

# 🟠 Solution: 2D Convolution（参考版）

## 📋 题目描述

**难度：** Medium

从零实现二维卷积。

卷积将卷积核在二维输入上滑动，在每个位置计算加权和，是 CNN 的基本操作。

**签名:** `my_conv2d(x, weight, bias=None, stride=1, padding=0) -> Tensor`

**参数:**
- `x` — 输入张量 (B, C_in, H, W)
- `weight` — 卷积核张量 (C_out, C_in, kH, kW)
- `bias` — 可选偏置 (C_out,)
- `stride`, `padding` — 整数步幅和零填充

**返回:** 卷积输出张量

**约束:**
- 必须与 `F.conv2d` 数值一致
- 支持 stride 和 padding 参数

**提示：** 1. 手动对输入进行零填充（不用 F.pad）
2. `x.unfold(2, kH, stride).unfold(3, kW, stride)` → 块 `(B, C, H_out, W_out, kH, kW)`
3. `(patches * weight).sum((-3,-2,-1)) + bias`


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


In [ ]:
# ✅ SOLUTION

def my_conv2d(x, weight, bias=None, stride=1, padding=0):
    # ---- Step 1: 零填充 ----
    # padding 在输入四周添加零行/列
    # [padding]*4 = [left, right, top, bottom] 都加 padding 个零
    # 例如 padding=1：H×W → (H+2)×(W+2)
    # F.pad 是 PyTorch 内置的填充函数
    if padding > 0:
        x = F.pad(x, [padding] * 4)

    B, C_in, H, W = x.shape
    C_out, _, kH, kW = weight.shape

    # ---- Step 2: 计算输出尺寸 ----
    # H_out = (H - kH) / stride + 1
    # 例如 H=32, kH=3, stride=1 → (32-3)/1+1 = 30
    # 例如 H=32, kH=3, stride=2 → (32-3)/2+1 = 15
    H_out = (H - kH) // stride + 1
    W_out = (W - kW) // stride + 1

    # ---- Step 3: 用 unfold 提取滑动窗口 ----
    # unfold(dim, size, step) 在指定维度上提取大小为 size、步长为 step 的窗口
    # x.unfold(2, kH, stride)：在 H 维度上提取 kH 大小的窗口
    #   (B, C_in, H, W) → (B, C_in, H_out, W, kH)
    # x.unfold(3, kW, stride)：在 W 维度上提取 kW 大小的窗口
    #   → (B, C_in, H_out, W_out, kH, kW)
    # 每个 (h,w) 位置的 patch shape = (C_in, kH, kW)
    patches = x.unfold(2, kH, stride).unfold(3, kW, stride)
    # patches shape: (B, C_in, H_out, W_out, kH, kW)

    # ---- Step 4: 用 einsum 计算卷积 ----
    # einsum 爱因斯坦求和约定：
    # 'bihwjk,oijk->bohw'
    #   b: batch, i: C_in, h,w: 输出位置, j: kH, k: kW, o: C_out
    # 对 i,j,k 求和 → 输出 (b,o,h,w)
    # 这等价于在每个输出位置做卷积核的点积
    out = torch.einsum('bihwjk,oijk->bohw', patches, weight)

    # ---- Step 5: 加偏置 ----
    # bias shape (C_out,) → (1, C_out, 1, 1) 以便广播到 (B, C_out, H_out, W_out)
    if bias is not None:
        out = out + bias.view(1, -1, 1, 1)

    return out


In [ ]:
x = torch.randn(1, 3, 8, 8)
w = torch.randn(16, 3, 3, 3)
b = torch.randn(16)
out = my_conv2d(x, w, b, stride=1, padding=1)
ref = F.conv2d(x, w, b, stride=1, padding=1)
print(f'Output: {out.shape}, Match: {torch.allclose(out, ref, atol=1e-5)}')


In [ ]:
from torch_judge import check
check('conv2d')


## 📝 核心思路总结

1. **unfold 提取窗口**：在 H/W 维度上提取滑动窗口，得到所有 patch
2. **einsum/matrox 乘法**：patch 与卷积核做点积，对 C_in×kH×kW 求和
3. **输出尺寸公式**：`H_out = (H - kH) // stride + 1`
4. **等价于 im2col**：unfold + reshape 就是经典的 im2col 方法
